# 训练子空间 fc head (H / L)

对 H (317维) 和 L (451维) 子空间各训练一个独立的 Linear(K→2) 分类头，
目标：次日涨跌二分类。训练完输出 class_1 prob 宽表 (date × symbol)。

- Walk-forward: train=2020, test=2021
- CrossEntropyLoss, Adam, 早停
- 不评估，只输出宽表供回测

In [ ]:
# 1. 加载 sum_cls + 投影到 H/L + 收益标签 + walk-forward
import os, sys, glob, numpy as np, pandas as pd, torch, torch.nn as nn
from sklearn.preprocessing import StandardScaler

ROOT = "/home/intern_fjq_2026/Projects/chinese-wwm-roberta"
os.chdir(ROOT); sys.path.insert(0, ROOT)
DEVICE = "cuda:1"

# --- 加载 per_file ---
PER_DIR = os.path.join(ROOT, "artifacts", "gubapost_cls", "per_file")
pf_files = sorted(glob.glob(os.path.join(PER_DIR, "*.parquet")))
assert pf_files, "per_file/ 下没有文件"
print(f"加载 {len(pf_files)} 个 per-file...")
dfs = [pd.read_parquet(f, columns=["available_date", "symbol", "n_posts", "sum_cls"]) for f in pf_files]
pf = pd.concat(dfs, ignore_index=True); del dfs
nposts = pf["n_posts"].values.astype(np.float64)
nposts_safe = np.where(nposts > 0, nposts, 1.0)
mean_cls = (np.stack(pf["sum_cls"].values).astype(np.float64) / nposts_safe[:, None]).astype(np.float32)
meta = pf[["available_date", "symbol", "n_posts"]].copy()
del pf
print(f"mean_cls: {mean_cls.shape} | dates: {meta.available_date.min()}~{meta.available_date.max()}")

# --- coverage H/L 方向 ---
RUN_DIR = os.path.join(ROOT, "artifacts", "checkpoint_activation_rank", "runs", "gubapost_v1")
cov = np.load(os.path.join(RUN_DIR, "extensions", "coverage_ablation_v1", "direction_sets.npz"))
Q_H = cov["keep_317_complement_K64"].astype(np.float32)   # [768, 317]
Q_L = cov["keep_451_lowcov_K64"].astype(np.float32)       # [768, 451]
# 紧凑特征: Q^T @ mean_cls
feat_H = mean_cls @ Q_H   # [N, 317]
feat_L = mean_cls @ Q_L   # [N, 451]
print(f"feat_H {feat_H.shape} feat_L {feat_L.shape}")

# --- 收益 (前移1天, winsorize, 涨跌标签) ---
rtn = pd.read_parquet("/home/intern_fjq_2026/data/RTN_daily/rtn_1d.parquet")
rtn_long = rtn.melt(id_vars="date", var_name="sym", value_name="r")
rtn_long["symbol"] = rtn_long["sym"].str.split(".").str[0]
rtn_long["date"] = pd.to_datetime(rtn_long["date"]).dt.strftime("%Y-%m-%d")
rtn_long = rtn_long.sort_values(["symbol", "date"])
rtn_long["y"] = rtn_long.groupby("symbol")["r"].shift(-1)
rtn_long = rtn_long[["date", "symbol", "y"]].dropna(subset=["y"])
rtn_long["y"] = rtn_long["y"].clip(-0.2, 0.2)
rtn_long["label"] = (rtn_long["y"] > 0).astype(np.int64)

# --- 对齐: meta 和 rtn 按 (date, symbol) join ---
merged = meta[["available_date", "symbol"]].merge(
    rtn_long, left_on=["available_date", "symbol"], right_on=["date", "symbol"], how="inner")
merged = merged[["available_date", "symbol", "y", "label"]].reset_index(drop=True)
merged["year"] = merged["available_date"].str[:4]

# 对齐 features
meta_key = meta["available_date"] + "_" + meta["symbol"]
merged_key = merged["available_date"] + "_" + merged["symbol"]
mask = meta_key.isin(set(merged_key)).values
feat_H = feat_H[mask]
feat_L = feat_L[mask]
nposts_aligned = nposts_safe[mask]
assert len(feat_H) == len(merged), f"{len(feat_H)} != {len(merged)}"

y_all = merged["y"].values.astype(np.float64)
label_all = merged["label"].values.astype(np.int64)
dates_all = merged["available_date"].values
years = sorted(merged["year"].unique())
MIN_TEST_DAYS = 20
folds = [(years[:i], years[i]) for i in range(1, len(years))
         if (merged["year"] == years[i]).sum() >= MIN_TEST_DAYS]
idx_all = np.arange(len(merged))
print(f"walk-forward: {len(folds)} folds, years={years}")
print(f"train samples: {(merged['year']==years[0]).sum():,}, test: {(merged['year']==years[1]).sum():,}")

In [ ]:
# 2. 训练 H 和 L 的 fc head
def train_fc_head(feat_train, label_train, feat_val, label_val, in_dim,
                 device=DEVICE, epochs=100, lr=1e-3, wd=1e-4, patience=5, bs=256):
    """训练一个 Linear(in_dim -> 2) 分类头, 返回训练好的模型."""
    model = nn.Linear(in_dim, 2).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    loss_fn = nn.CrossEntropyLoss()
    
    sc = StandardScaler()
    X_tr = torch.tensor(sc.fit_transform(feat_train), dtype=torch.float32, device=device)
    y_tr = torch.tensor(label_train, dtype=torch.long, device=device)
    X_val = torch.tensor(sc.transform(feat_val), dtype=torch.float32, device=device)
    y_val = torch.tensor(label_val, dtype=torch.long, device=device)
    
    n = len(X_tr)
    best_val_loss = float("inf")
    best_state = None
    no_improve = 0
    
    for ep in range(epochs):
        model.train()
        perm = torch.randperm(n, device=device)
        ep_loss = 0
        for i in range(0, n, bs):
            idx = perm[i:i+bs]
            logits = model(X_tr[idx])
            loss = loss_fn(logits, y_tr[idx])
            opt.zero_grad()
            loss.backward()
            opt.step()
            ep_loss += loss.item() * len(idx)
        
        model.eval()
        with torch.no_grad():
            val_loss = loss_fn(model(X_val), y_val).item()
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                break
    
    model.load_state_dict(best_state)
    model.eval()
    return model, sc

def predict_prob(model, sc, feat):
    """返回 class_1 概率 (numpy)."""
    X = torch.tensor(sc.transform(feat), dtype=torch.float32, device=DEVICE)
    with torch.no_grad():
        prob = torch.softmax(model(X), dim=1)[:, 1]
    return prob.cpu().numpy()

results = []
for fi, (tr, te) in enumerate(folds):
    ti = idx_all[merged["year"].isin(tr).values]
    ei = idx_all[(merged["year"] == te).values]
    print(f"\n=== Fold {tr} -> {te} ===")
    
    # train/val split (90/10)
    rng = np.random.default_rng(42)
    perm = rng.permutation(len(ti))
    n_val = max(1, len(ti) // 10)
    val_idx = ti[perm[:n_val]]
    train_idx = ti[perm[n_val:]]
    
    for name, feat, in_dim in [("H", feat_H, 317), ("L", feat_L, 451)]:
        model, sc = train_fc_head(
            feat[train_idx], label_all[train_idx],
            feat[val_idx], label_all[val_idx],
            in_dim=in_dim
        )
        # 预测全量 (train + test), 但只存 test
        prob_test = predict_prob(model, sc, feat[ei])
        results.append(pd.DataFrame({
            "date": dates_all[ei],
            "symbol": merged.iloc[ei]["symbol"].values,
            f"prob_{name}": prob_test.astype(np.float32),
            f"n_posts": nposts_aligned[ei].astype(np.int32),
        }))
        print(f"  {name}: trained {len(train_idx)} samples, predicted {len(ei)} test rows")

print("\n训练完成")

In [ ]:
# 3. 输出宽表 (date × symbol)
OUT_DIR = os.path.join(ROOT, "artifacts", "gubapost_cls", "trained_subspace_heads")
os.makedirs(OUT_DIR, exist_ok=True)

# 合并所有 fold 结果
all_results = pd.concat(results, ignore_index=True)
print(f"总计: {len(all_results):,} stock-days")

# H 宽表
wide_H = all_results.pivot_table(index="date", columns="symbol", values="prob_H", aggfunc="mean")
wide_H = wide_H.sort_index()
wide_H.to_parquet(os.path.join(OUT_DIR, "trained_fc_prob_H.parquet"))
print(f"trained_fc_prob_H: {wide_H.shape}")

# L 宽表
wide_L = all_results.pivot_table(index="date", columns="symbol", values="prob_L", aggfunc="mean")
wide_L = wide_L.sort_index()
wide_L.to_parquet(os.path.join(OUT_DIR, "trained_fc_prob_L.parquet"))
print(f"trained_fc_prob_L: {wide_L.shape}")

# long format
all_results.to_parquet(os.path.join(OUT_DIR, "trained_fc_prob_long.parquet"), index=False)
print(f"\n保存到: {OUT_DIR}")
for f in sorted(os.listdir(OUT_DIR)):
    sz = os.path.getsize(os.path.join(OUT_DIR, f))
    print(f"  {f}: {sz/1e6:.1f}MB")

In [ ]:
# 4. 预览
print("=== trained_fc_prob_H ===")
print(wide_H.iloc[:3, :4])
print(f"  shape: {wide_H.shape}, NaN: {wide_H.isna().sum().sum()/wide_H.size:.1%}")
print(f"  range: [{wide_H.min().min():.4f}, {wide_H.max().max():.4f}]")
print()
print("=== trained_fc_prob_L ===")
print(wide_L.iloc[:3, :4])
print(f"  shape: {wide_L.shape}, NaN: {wide_L.isna().sum().sum()/wide_L.size:.1%}")
print(f"  range: [{wide_L.min().min():.4f}, {wide_L.max().max():.4f}]")
print("\n可以用来做回测。")